In [1]:
import sys
import pprint
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
from utils.json_utils import safe_parse_json, validate_json_schema, create_simple_schema, format_schema_for_prompt
from utils.config_loader import get_config,reload_config
from pathlib import Path
reload_config()

In [2]:
from pydantic import BaseModel, field_validator

class CrisisEvent(BaseModel):
    district: str
    flood_level_meters: float | None = None
    victim_count: int = 0
    main_need: str
    status: str

In [3]:
from utils.config_loader import load_config

config = load_config()

VALID_DISTRICTS = config.get("classification.districts")

print(VALID_DISTRICTS)

['Ampara', 'Anuradhapura', 'Badulla', 'Batticaloa', 'Colombo', 'Galle', 'Gampaha', 'Hambantota', 'Jaffna', 'Kalutara', 'Kandy', 'Kegalle', 'Kilinochchi', 'Kurunegala', 'Mannar', 'Matale', 'Matara', 'Monaragala', 'Mullaitivu', 'Nuwara Eliya', 'Polonnaruwa', 'Puttalam', 'Ratnapura', 'Trincomalee', 'Vavuniya']


In [4]:
from pydantic import BaseModel, field_validator
from typing import Literal

class CrisisEvent(BaseModel):
    district: str | None
    flood_level_meters: float | None = None
    victim_count: int = 0
    main_need: str
    status: Literal[
        "Critical",
        "Warning",
        "Stable"
    ]

    @field_validator("district")
    @classmethod
    def validate_district(cls, value):
        if value == "Unknown":
            return value
        if value not in VALID_DISTRICTS:
            raise ValueError(
                f"Invalid district: {value}"
            )

        return value

Test the schema 

In [5]:
test_json = """
{
    "district": "Gampaha",
    "flood_level_meters": null,
    "victim_count": 5,
    "main_need": "Rescue boat",
    "status": "Critical"
}
"""

event = CrisisEvent.model_validate_json(test_json)

print(event)

district='Gampaha' flood_level_meters=None victim_count=5 main_need='Rescue boat' status='Critical'


Load the news feed

In [6]:
news_path = Path("../data/news_feed.txt")

with open(
    news_path,
    "r",
    encoding="utf-8"
) as file:

    news_items = [
        line.strip()
        for line in file
        if line.strip()
    ]

print("Total news items:", len(news_items))

for index, item in enumerate(news_items, start=1):
    print(index, item)


Total news items: 30
1 BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.
2 SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
3 Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.
4 URGENT: Landslide in Kalutara. 12 people missing. Rescue team needed.
5 Gampaha town center is fully underwater. Flood level est 2.0 meters. 500 people displaced to temple. Need dry rations.
6 Just saw a navy boat in Colombo. Good job guys.
7 Matara reports heavy rain but no floods yet. Status stable.
8 Help! My grandmother is stuck in Beddagana (Colombo). She is 80 years old. Water level 1.5m.
9 Galle fort area is safe. Tourists are fine.
10 Warning: Dengue risk rising in Gampaha due to stagnant water.
11 SOS: Kaduwela highway entrance (Colombo) blocked. Bus trapped with 40 passengers. Need evacuation.
12 Donation drive starting at Town Hall. We need water bottles and biscu

Select the model

In [7]:
model = pick_model(
    provider="groq",
    technique="general"
)

print("Model:", model)

llm = LLMClient(
    provider="groq",
    model=model
)

Model: openai/gpt-oss-20b


In [8]:
def extract_crisis_event(message, llm):

    district_text = ", ".join(VALID_DISTRICTS)

    prompt_text, _ = render(
        "json_extract.v1",

        role="crisis information extraction assistant",

        query=message,

        instruction="""
Extract the crisis-related information from the input message
into the required JSON structure.
""",

        schema="""
{
    "district": string,
    "flood_level_meters": float or null,
    "victim_count": integer,
    "main_need": string,
    "status": "Critical" | "Warning" | "Stable"
}
""",

        constraints=f"""
Valid districts are:
{district_text}

Rules:

1. Use only information stated or directly supported by the message.

2. Do not invent a district.
   If no valid district can be identified, use "Unknown".
   Pydantic validation will reject that record later.

3. flood_level_meters:
   - Use the numerical flood/water level if explicitly stated.
   - Otherwise use null.

4. victim_count:
   - Use an explicitly stated number of victims/affected people
     when appropriate.
   - If no victim count is stated, use 0.
   - Do not invent a victim count.

5. main_need:
   - Extract the stated main need.
   - If no specific need is stated, use "None".

6. status:
   - Critical: immediate danger, rescue, trapped/missing/injured people,
     or serious emergency.
   - Warning: risk or dangerous condition requiring attention but no
     immediate rescue described.
   - Stable: safe, normal, cleared, or no immediate danger reported.

7. Return exactly one JSON object.

8. Do not include Markdown code fences.

9. Do not include explanations before or after the JSON.
""",

        format="""
{
    "district": "...",
    "flood_level_meters": null,
    "victim_count": 0,
    "main_need": "...",
    "status": "..."
}
"""
    )

    response = llm.chat(
        [
            {
                "role": "user",
                "content": prompt_text
            }
        ],
        temperature=0.0,
        task_type="extraction"
    )

    return response["text"].strip()

Test

In [9]:
test_message = news_items[1]

print("Input:")
print(test_message)

json_output = extract_crisis_event(
    test_message,
    llm
)

print("\nExtracted JSON:")
print(json_output)

Input:
SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.

Extracted JSON:
{
    "district": "Gampaha",
    "flood_level_meters": null,
    "victim_count": 5,
    "main_need": "boat",
    "status": "Critical"
}


In [10]:
try:

    validated_event = CrisisEvent.model_validate_json(
        json_output
    )

    print("VALID")
    print(validated_event)

except ValidationError as error:

    print("INVALID")
    print(error)

VALID
district='Gampaha' flood_level_meters=None victim_count=5 main_need='boat' status='Critical'


Process all 30 items

In [11]:
valid_events = []
invalid_events = []

for index, message in enumerate(
    news_items,
    start=1
):

    print("=" * 70)
    print(f"Processing item {index}")
    print(message)

    try:

        json_output = extract_crisis_event(
            message,
            llm
        )

        event = CrisisEvent.model_validate_json(
            json_output
        )

        valid_events.append(event)

        print("VALID")
        print(event)

    except Exception as error:

        invalid_events.append({
            "item": index,
            "message": message,
            "error": str(error)
        })

        print(
            f"WARNING: Item {index} is invalid and was skipped."
        )
        print(error)

Processing item 1
BREAKING: Water levels in Kelani River (Colombo) have reached 9.5 meters. Critical flood warning issued.
VALID
district='Colombo' flood_level_meters=9.5 victim_count=0 main_need='None' status='Critical'
Processing item 2
SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
VALID
district='Gampaha' flood_level_meters=None victim_count=5 main_need='boat' status='Critical'
Processing item 3
Update: Kandy road cleared near Peradeniya. Traffic moving slowly. No victims reported.
VALID
district='Kandy' flood_level_meters=None victim_count=0 main_need='None' status='Stable'
Processing item 4
URGENT: Landslide in Kalutara. 12 people missing. Rescue team needed.
VALID
district='Kalutara' flood_level_meters=None victim_count=12 main_need='Rescue team' status='Critical'
Processing item 5
Gampaha town center is fully underwater. Flood level est 2.0 meters. 500 people displaced to temple. Need dry rations.
VALID
district='Gampaha' flood_le

Check the result counts

In [12]:
print("=" * 50)
print("EXTRACTION SUMMARY")
print("=" * 50)

print("Total input items :", len(news_items))
print("Valid events      :", len(valid_events))
print("Invalid events    :", len(invalid_events))

EXTRACTION SUMMARY
Total input items : 30
Valid events      : 30
Invalid events    : 0


Inspect invalid records

In [13]:
for item in invalid_events:

    print("=" * 70)
    print("Item:", item["item"])
    print("Message:", item["message"])
    print("Error:", item["error"])

Convert valid Pydantic objects to dictionaries

In [14]:
event_records = [
    event.model_dump()
    for event in valid_events
]

print(event_records[:3])

[{'district': 'Colombo', 'flood_level_meters': 9.5, 'victim_count': 0, 'main_need': 'None', 'status': 'Critical'}, {'district': 'Gampaha', 'flood_level_meters': None, 'victim_count': 5, 'main_need': 'boat', 'status': 'Critical'}, {'district': 'Kandy', 'flood_level_meters': None, 'victim_count': 0, 'main_need': 'None', 'status': 'Stable'}]


Create Pandas DataFrame

In [15]:
import pandas as pd
df = pd.DataFrame(event_records)

df.head()

,district,flood_level_meters,victim_count,main_need,status
0,Colombo,9.5,0,None,Critical
1,Gampaha,NaN,5,boat,Critical
2,Kandy,NaN,0,None,Stable
3,Kalutara,NaN,12,Rescue team,Critical
4,Gampaha,2.0,500,dry rations,Critical


In [16]:
print(df.columns.tolist())
print("Rows:", len(df))

['district', 'flood_level_meters', 'victim_count', 'main_need', 'status']
Rows: 30


Save to Excel

In [17]:
output_path = Path(
    "../output/flood_report.xlsx"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_excel(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

Saved to: ..\output\flood_report.xlsx
